# Notebook 01 — Preprocessing Evaluation

Evaluates Module 1 (Preprocessor):
1. Artificially degrade 20 clean images (rotate, add noise, reduce contrast)
2. Run TrOCR on raw vs preprocessed versions
3. Report before/after CER delta per step
4. Generate before/after image gallery

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import jiwer
from PIL import Image
from pathlib import Path
import random

from src.preprocessing import Preprocessor
from src.ocr import TrOCREngine

print('Imports OK')

In [ ]:
# ── Degradation Functions ─────────────────────────────────────────────────────

def add_rotation(img_arr, angle=15):
    h, w = img_arr.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img_arr, M, (w, h), borderValue=255)

def add_noise(img_arr, sigma=25):
    noise = np.random.normal(0, sigma, img_arr.shape).astype(np.int16)
    return np.clip(img_arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def reduce_contrast(img_arr, factor=0.4):
    mid = 128
    return np.clip(mid + (img_arr.astype(np.float32) - mid) * factor, 0, 255).astype(np.uint8)

def combined_degradation(img_arr):
    out = add_rotation(img_arr, 12)
    out = add_noise(out, 30)
    out = reduce_contrast(out, 0.5)
    return out

print('Degradation functions defined.')

In [ ]:
# ── Load 20 IAM samples & degrade ────────────────────────────────────────────
IAM_DIR = Path('../data/iam')

# Load samples (reuse code from notebook 00)
samples = []  # populate from words.txt as in notebook 00
# ... (copy loading code from 00_data_exploration.ipynb)

subset = random.sample(samples, 20)
results = []

engine = TrOCREngine()

for s in subset:
    img = Image.open(s['path']).convert('RGB')
    arr = np.array(img)
    degraded = combined_degradation(arr)
    results.append({
        'label': s['label'],
        'original': img,
        'degraded': Image.fromarray(degraded),
    })

print(f'Prepared {len(results)} degraded samples.')

In [ ]:
# ── Evaluate: raw degraded vs preprocessed ───────────────────────────────────
configs = {
    'Raw (degraded)':     {'deskew': False, 'denoise': False, 'contrast': False, 'binarize': False},
    'Deskew only':        {'deskew': True,  'denoise': False, 'contrast': False, 'binarize': False},
    'Denoise only':       {'deskew': False, 'denoise': True,  'contrast': False, 'binarize': False},
    'CLAHE only':         {'deskew': False, 'denoise': False, 'contrast': True,  'binarize': False},
    'Sauvola only':       {'deskew': False, 'denoise': False, 'contrast': False, 'binarize': True},
    'Full pipeline':      {'deskew': True,  'denoise': True,  'contrast': True,  'binarize': True},
}

cer_results = {}
for name, config in configs.items():
    pre = Preprocessor(config=config)
    preds, refs = [], []
    for r in results:
        cleaned = pre.transform(r['degraded'])
        candidates = engine.recognise(cleaned)
        preds.append(candidates[0].word if candidates else '')
        refs.append(r['label'])
    cer = jiwer.cer(refs, preds)
    cer_results[name] = cer
    print(f'{name:30s} CER: {cer*100:.1f}%')

print('\nDone. Lower CER = better preprocessing for these degraded samples.')